# Promising HOG Pipeline Combinations (Full PlantVillage)

This notebook evaluates 3 selected high-potential combinations on the **full PlantVillage dataset (all classes)** using:
- Preprocessing: `gaussian`
- Segmentation: `otsu`
- Edge: `canny`
- Corner: `shi_tomasi`
- Feature: `hog`

Morphology combinations tested:
1. `opening` only
2. `erosion` then `dilation`
3. `opening` then `closing`

In [1]:
from pathlib import Path

import cv2
import numpy as np

from classifiers import train_and_evaluate
from features import detect_corners, detect_edges, extract_hog
from processing import apply_morphology, apply_preprocessing, segment_leaf

In [ ]:
dataset_root = Path("PlantVillage")
if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset root not found: {dataset_root}")

fixed_preprocess = "gaussian"
fixed_segmentation = "otsu"
fixed_edge = "canny"
fixed_corner = "shi_tomasi"
fixed_feature = "hog"

morphology_configs = [
    ("opening", None, "opening only"),
    ("erosion", "dilation", "erosion then dilation"),
    ("opening", "closing", "opening then closing"),
]

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def list_class_dirs(root: Path) -> list[Path]:
    class_dirs = []
    for candidate in sorted(root.iterdir()):
        if not candidate.is_dir():
            continue
        if candidate.name == "PlantVillage":
            continue
        has_images = any(
            image_path.is_file() and image_path.suffix.lower() in image_extensions
            for image_path in candidate.rglob("*")
        )
        if has_images:
            class_dirs.append(candidate)
    return class_dirs


def list_images(class_dir: Path) -> list[Path]:
    images = [
        path for path in class_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    ]
    return sorted(images)


def process_for_hog(
    grayscale: np.ndarray,
    preprocess: str,
    segmentation: str,
    morph_first: str | None,
    morph_second: str | None,
    edge: str,
    corner: str,
) -> np.ndarray:
    working = apply_preprocessing(grayscale, filter_type=preprocess)

    mask = segment_leaf(working, method=segmentation)
    if morph_first is not None:
        mask = apply_morphology(mask, operation=morph_first)
    if morph_second is not None:
        mask = apply_morphology(mask, operation=morph_second)

    working = cv2.bitwise_and(working, working, mask=mask)
    working = detect_edges(working, method=edge)
    corners_bgr = detect_corners(working, method=corner)
    final_gray = cv2.cvtColor(corners_bgr, cv2.COLOR_BGR2GRAY)

    return final_gray


def build_dataset_for_combo(class_dirs: list[Path], morph_first: str | None, morph_second: str | None) -> tuple[np.ndarray, np.ndarray]:
    X_rows = []
    y_rows = []

    total_images = 0
    for class_dir in class_dirs:
        image_paths = list_images(class_dir)
        print(f"Class: {class_dir.name} | images: {len(image_paths)}")

        for image_path in image_paths:
            image_bgr = cv2.imread(str(image_path))
            if image_bgr is None:
                continue

            grayscale = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
            processed = process_for_hog(
                grayscale=grayscale,
                preprocess=fixed_preprocess,
                segmentation=fixed_segmentation,
                morph_first=morph_first,
                morph_second=morph_second,
                edge=fixed_edge,
                corner=fixed_corner,
            )

            hog_features, _ = extract_hog(processed)
            X_rows.append(hog_features.astype(np.float32))
            y_rows.append(class_dir.name)
            total_images += 1

            if total_images % 500 == 0:
                print(f"  Processed {total_images} images so far...")

    if len(X_rows) < 5:
        raise ValueError("Need at least 5 valid samples for 5-fold CV.")

    X = np.vstack(X_rows).astype(np.float32)
    y = np.array(y_rows)
    return X, y

In [7]:
class_directories = list_class_dirs(dataset_root)
class_directories = [d for d in class_directories if d != "PlantVillage"]
if not class_directories:
    raise ValueError(f"No class folders with images found in {dataset_root}.")

print(f"Found {len(class_directories)} classes.")
for class_dir in class_directories:
    print(f"- {class_dir.name}")

Found 16 classes.
- Pepper__bell___Bacterial_spot
- Pepper__bell___healthy
- PlantVillage
- Potato___Early_blight
- Potato___healthy
- Potato___Late_blight
- Tomato__Target_Spot
- Tomato__Tomato_mosaic_virus
- Tomato__Tomato_YellowLeaf__Curl_Virus
- Tomato_Bacterial_spot
- Tomato_Early_blight
- Tomato_healthy
- Tomato_Late_blight
- Tomato_Leaf_Mold
- Tomato_Septoria_leaf_spot
- Tomato_Spider_mites_Two_spotted_spider_mite


In [ ]:
all_results = []

for morph_first, morph_second, combo_name in morphology_configs:
    print("\n" + "=" * 90)
    print(f"Evaluating combo: {combo_name}")
    print("=" * 90)

    X, y = build_dataset_for_combo(class_directories, morph_first, morph_second)
    print(f"Feature matrix shape: {X.shape}")
    print(f"Labels shape: {y.shape}")

    combo_metrics = train_and_evaluate(X, y)

    for model_name, metrics in combo_metrics.items():
        print(f"Model: {model_name}")
        print(f"  Accuracy: {metrics['accuracy']:.4f}")
        print(f"  Weighted F1-score: {metrics['f1_score']:.4f}")

        all_results.append(
            {
                "combo": combo_name,
                "morph_first": morph_first,
                "morph_second": morph_second,
                "model": model_name,
                "accuracy": float(metrics["accuracy"]),
                "weighted_f1": float(metrics["f1_score"]),
                "confusion_matrix": metrics["confusion_matrix"],
                "confusion_matrix_figure": metrics["confusion_matrix_figure"],
            }
        )

In [ ]:
print("\nFinal ranking by weighted F1-score:")
sorted_results = sorted(all_results, key=lambda result: result["weighted_f1"], reverse=True)
for rank, result in enumerate(sorted_results, start=1):
    print(
        f"{rank:02d}. combo={result['combo']} | model={result['model']} | "
        f"acc={result['accuracy']:.4f} | f1={result['weighted_f1']:.4f}"
    )